In [0]:
# import pandas as pd

# # Create a small sample CSV file
# data = {
#     "id": [1, 2, 3, 4, 5],
#     "name": ["Alice", "Bob", "Carol", "Dave", "Eve"],
#     "age": [25, 30, 35, 40, 45],
#     "city": ["Seattle", "Portland", "Denver", "Austin", "Boston"],
# }
# csv_path = "/tmp/sample_data.csv"
# pd.DataFrame(data).to_csv(csv_path, index=False)
# print(f"CSV written to {csv_path}")

# # Read the CSV back and display it
# df = pd.read_csv(csv_path)
# display(df)

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.\
                builder.\
                    appName("Python Spark SQL basic example").\
                        getOrCreate()

In [0]:
## reading a csv file
csv_path = "/Volumes/pyspark/data/pyspark/employee.csv"
df = spark.read.format("csv").\
                load(csv_path)
                
# display(df)
df.printSchema()

In [0]:
csv_path = "/Volumes/pyspark/data/pyspark/employee.csv"
df = spark.read.format("csv").\
                option("header", "true").\
                    load(csv_path)
                
df.printSchema()

In [0]:
csv_path = "/Volumes/pyspark/data/pyspark/employee.csv"
df = spark.read.format("csv").\
                option("header", "true").\
                    option("inferSchema", "true").\
                        load(csv_path)
                
df.printSchema()

#### Carefully observe the above output and understand the difference and uses of all the triggers and their param.

In [0]:
df.show()

In [0]:
## reading a csv file with schema provided explicitly (DDL string format)
csv_path = "/Volumes/pyspark/data/pyspark/employee.csv"

_schema = "emp_id INT, emp_name STRING, department STRING, salary INT, age INT, city STRING, joining_year INT, manager_id INT"

df = spark.read.format("csv").\
                option("header", "true").\
                    schema(_schema).\
                        load(csv_path)
                
df.show()

In [0]:
# ## creating a csv file with some bad/corrupt records
# ## (wrong data types, missing fields, extra fields) so we can practice handling bad records

# _header = "emp_id,emp_name,department,salary,age,city,joining_year,manager_id"

# _rows = [
#     "1,Alice,Engineering,75000,25,Seattle,2020,3",        # good record
#     "2,Bob,Sales,55000,30,Portland,2019,",                 # good record (nullable manager_id)
#     "three,Carol,HR,not_a_salary,35,Denver,2021,5",       # bad: emp_id and salary are non-numeric
#     "4,Dave",                                              # bad: missing most columns
#     "5,Eve,Austin",                                       # bad: wrong column count / shifted values
#     "6,Frank,Marketing,62000,twenty,Boston,2022,7",       # bad: age is non-numeric
#     "7,Grace,Engineering,80000,29,Austin,2018,abc",       # bad: manager_id is non-numeric
#     "8,Heidi,Finance,70000,45,Boston,2017,3,extra_value",  # bad: extra column
#     "9,Ivan,Finance,65000,38,Chicago,2023,3",             # good record
# ]

# csv_path = "/Volumes/pyspark/data/pyspark/employee_bad_records.csv"

# with open(csv_path, "w") as f:
#     f.write(_header + "\n")
#     f.write("\n".join(_rows) + "\n")

# print(f"CSV with bad records written to {csv_path}")

# # preview the file contents
# with open(csv_path) as f:
#     print(f.read())

In [0]:
## Handling bad records
# default mode PERMISSIVE
csv_path = "/Volumes/pyspark/data/pyspark/employee_bad_records.csv"
_schema = "emp_id INT, emp_name STRING, department STRING, salary INT, age INT, city STRING, joining_year INT, manager_id INT, bad_records STRING"

df = spark.read.format("csv").\
                option("header", "true").\
                    schema(_schema).\
                        option("columnNameOfCorruptRecord", "bad_records").\
                            load(csv_path)

df.printSchema()
df.show()

In [0]:
## Handling bad records DROPMALFORMED
# drop the bad records
csv_path = "/Volumes/pyspark/data/pyspark/employee_bad_records.csv"
_schema = "emp_id INT, emp_name STRING, department STRING, salary INT, age INT, city STRING, joining_year INT, manager_id INT"

df_m = spark.read.format("csv").\
                option("header", "true").\
                    schema(_schema).\
                        option("mode", "DROPMALFORMED").\
                            load(csv_path)

df_m.printSchema()
df_m.show()

In [0]:
## Handling bad records FAILFAST
# It fails to read the file if we encounter any bad records
csv_path = "/Volumes/pyspark/data/pyspark/employee_bad_records.csv"
_schema = "emp_id INT, emp_name STRING, department STRING, salary INT, age INT, city STRING, joining_year INT, manager_id INT"

df_f = spark.read.format("csv").\
                option("header", "true").\
                    schema(_schema).\
                        option("mode", "FAILFAST").\
                            load(csv_path)

df_f.printSchema()
df_f.show()

In [0]:
## Multiple options
csv_path = "/Volumes/pyspark/data/pyspark/employee_bad_records.csv"
_option = {
    "inferSchema": "true",
    "header": "true",
    'mode': "PERMISSIVE"
}

df = spark.read.format("csv").\
            options(**_option).\
                load(csv_path)
df.show()


### ROW vs COLUMNAR file format

#### Row-oriented (row-based) format — e.g. CSV, JSON, Avro
- Data is stored **one record (row) at a time**, with all of a record's fields together on the same line/structure.
- To read a single column (e.g. `salary`), the engine must **scan and parse every row**, reading all the other columns too.
- No built-in schema, compression, or statistics — every read is a full pass over the file (we saw this with `inferSchema`, which triggers an extra job to guess types).
- Works well when you always need **all columns of a record** (e.g. transactional/OLTP workloads), or for simple human-readable interchange.
- Poor for analytics: aggregating one column over millions of rows means reading gigabytes of data you don't need.

#### Column-oriented (columnar) format — e.g. Parquet, ORC, Delta Lake
- Data is stored **column by column**: all values of `emp_id` together, then all values of `emp_name`, and so on.
- **Predicate pushdown**: filtering on a column (e.g. `salary > 60000`) touches only that column's data.
- **Projection pushdown**: queries reading 2 of 8 columns only read those 2 columns' bytes — massive I/O savings.
- Self-describing: **schema is embedded** in the file (no `inferSchema` needed), along with per-column **statistics** (min/max/count) for skipping row groups, plus efficient **encoding and compression** (run-length, dictionary) because values in a column share a type.
- Ideal for **analytical/OLAP workloads** — wide tables, aggregations, selective column reads.

#### Quick comparison
| Aspect | Row (CSV/JSON/Avro) | Columnar (Parquet/ORC/Delta) |
|---|---|---|
| Storage unit | Entire record | Entire column |
| Read 1 column of 8 | Reads all 8 columns | Reads only 1 column |
| Schema | External / inferred | Embedded in file |
| Compression | Weak (mixed types per row) | Strong (similar values together) |
| Best for | OLTP, full-record access | OLAP, aggregation & selective reads |

In the notebook so far we used **CSV (row format)** — simple and human-readable, but we had to fight with schemas, headers, and bad records. Switching the same data to a **columnar format** would avoid all of that and speed up analytical queries.



### Parquet vs ORC vs Avro — format comparison

All three are **self-describing, schema-embedded, binary formats** built for the Hadoop/Spark ecosystem — but they optimize for very different access patterns.

| Aspect | Parquet | ORC | Avro |
|---|---|---|---|
| Orientation | **Columnar** | **Columnar** | **Row-based** |
| Best for | Analytical reads (OLAP), selective column scans | Heavy analytical scans with predicate pushdown | Streaming / write-heavy, event logs, ETL interchange |
| Schema storage | Embedded in footer, per column chunk | Embedded, per-stripe index + footer | Embedded in the **header of every file** |
| Schema evolution | Good — add columns at end | Good — add columns | **Excellent** — fields resolve by name, add/rename easily |
| Compression | Excellent (Snappy/Zstd; dictionary + RLE) | Excellent (Zlib default; often best compression ratio) | Modest (Snappy/Deflate; mixed types per row) |
| Split-ability (parallel reads) | Splittable (row groups) | Splittable (stripes) | Splittable |
| Statistics / indexes | Min/max per column chunk, row-group skipping | Rich — stripe indexes, bloom filters, min/max | None built in |
| Encoding | Dictionary, RLE, delta encoding | Dictionary, RLE, lightweight indexes | Binary (no column encodings) |
| Row-level writes/appends | Weak (designed for batch) | Weak (designed for batch) | **Strong** — optimized for fast sequential writes |
| Complex / nested types | Very good support | Good support | **Best** — designed for complex schemas |
| Ecosystem | Default in **Spark, Delta Lake**, Hive; broadest tooling | Default in Hive/HBase ecosystems | Kafka Connect, Hadoop serialization, RPC, pipelines |
| When to choose | General analytics on Spark/Databricks (default choice) | Hive-centric, compression-critical, huge table scans | Ingestion layer: streaming events, evolving schemas |

#### Rule of thumb
- **Parquet** → the default for analytics on Spark/Databricks, and the foundation of Delta Lake.
- **ORC** → when you live in Hive and need the tightest compression on very large tables.
- **Avro** → when records arrive as a stream and schemas evolve frequently (ingestion), typically converted to Parquet/ORC for querying.



In [0]:
# ## create a small dataset (20 records) and write it as a Parquet file
# import random

# data = [
#     (i,
#      f"emp_{i:02d}",
#      random.choice(["Engineering", "Sales", "HR", "Finance", "Marketing"]),
#      random.randint(50000, 95000),
#      random.randint(22, 55),
#      random.choice(["Seattle", "Portland", "Denver", "Austin", "Boston"]),
#      random.randint(2015, 2023),
#      random.choice([1, 2, 3, 4, 5]))
#     for i in range(1, 21)  # 20 records
# ]

# _schema = "emp_id INT, emp_name STRING, department STRING, salary INT, age INT, city STRING, joining_year INT, manager_id INT"

# df = spark.createDataFrame(data, schema=_schema)

# parquet_path = "/Volumes/pyspark/data/pyspark/employee_20.parquet"
# df.write.format("parquet").mode("overwrite").save(parquet_path)

# print(f"Parquet file written to {parquet_path}")

In [0]:
## reading parquet file
parquet_path = "/Volumes/pyspark/data/pyspark/employee_20.parquet"

df_parq = spark.read.format("parquet").\
                load(parquet_path)

df_parq.printSchema()
df_parq.show()

#### Time taken

In [0]:
import time

def get_time(func):
    def inner_time() -> str:
        start = time.time()
        func()
        end = time.time()
        return f"Execution time: {(end - start) * 1000} ms"
    print(inner_time())

In [0]:
parquet_path = "/Volumes/pyspark/data/pyspark/employee_20.parquet"

@get_time
def read_parquet():
    df_parq = spark.read.format("parquet").\
                    load(parquet_path)
    df_parq.count()

In [0]:
@get_time
def read_parquet():
    df_parq = spark.read.format("parquet").\
                    load(parquet_path)
    df_parq.select("city").count()

#### Recursive file readewr

* To read files which are inside the folder reursively
    - folder/subfolder1/file1
    - folder/subfolder2/file2

* we can set `recursiveFileLookup` option as true